# 01. Pozyskanie i opis danych

**Etap z planu pracy:** pozyskanie danych.

Celem jest opis zrodla, struktury, liczby obserwacji i potencjalnych problemow jakosciowych. Baza uzywana dalej jest juz przygotowana, dlatego ten notebook nie wykonuje ponownego czyszczenia.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205_prepared.csv"
RAW_DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
METADATA_PATH = PROJECT_ROOT / "outputs" / "prepared_dataset_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "czysta_baza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Prepared data exists:", DATA_PATH.exists())
print("Output dir:", OUTPUT_DIR)

assert DATA_PATH.exists(), f"Brakuje pliku z przygotowana baza: {DATA_PATH}"


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH: C:\Users\szymon\projekt_reddit\database\NajnowszaWersjaBazy1205_prepared.csv
Prepared data exists: True
Output dir: C:\Users\szymon\projekt_reddit\outputs\czysta_baza


In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["TIMESTAMP"])
raw_exists = RAW_DATA_PATH.exists()

overview = pd.DataFrame([
    {"metric": "prepared_path", "value": str(DATA_PATH.relative_to(PROJECT_ROOT))},
    {"metric": "raw_path", "value": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)) if raw_exists else "brak"},
    {"metric": "rows", "value": df.shape[0]},
    {"metric": "columns", "value": df.shape[1]},
    {"metric": "timestamp_min", "value": df["TIMESTAMP"].min()},
    {"metric": "timestamp_max", "value": df["TIMESTAMP"].max()},
    {"metric": "unique_source_subreddits", "value": df["SOURCE_SUBREDDIT"].nunique()},
    {"metric": "unique_target_subreddits", "value": df["TARGET_SUBREDDIT"].nunique()},
    {"metric": "unique_directed_pairs", "value": df[["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT"]].drop_duplicates().shape[0]},
    {"metric": "negative_links", "value": int(df["is_negative_link"].sum())},
    {"metric": "negative_rate", "value": float(df["is_negative_link"].mean())},
])
display(overview)
overview.to_csv(OUTPUT_DIR / "01_data_overview.csv", index=False)


,metric,value
0,prepared_path,database\NajnowszaWersjaBazy1205_prepared.csv
1,raw_path,database\NajnowszaWersjaBazy1205.csv
2,rows,49918
3,columns,106
4,timestamp_min,2013-12-31 16:39:58
5,timestamp_max,2017-04-19 00:15:44
6,unique_source_subreddits,8605
7,unique_target_subreddits,6571
8,unique_directed_pairs,29621
9,negative_links,3854


In [3]:
split_quality = (
    df.groupby("split_chronological")["is_negative_link"]
    .agg(rows="size", negative_links="sum", negative_rate="mean")
    .reset_index()
)

sentiment_distribution = (
    df.groupby(["LINK_SENTIMENT", "Content_Sentiment"])
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(split_quality)
display(sentiment_distribution.head(12))
split_quality.to_csv(OUTPUT_DIR / "01_split_quality.csv", index=False)
sentiment_distribution.to_csv(OUTPUT_DIR / "01_sentiment_distribution.csv", index=False)


,split_chronological,rows,negative_links,negative_rate
0,test,9984,680,0.0681
1,train,39934,3174,0.0795


,LINK_SENTIMENT,Content_Sentiment,rows
4,1,0,32037
5,1,1,7658
3,1,-1,6369
1,-1,0,2146
0,-1,-1,1499
2,-1,1,209


In [4]:
key_columns = [
    "POST_ID",
    "TIMESTAMP",
    "SOURCE_SUBREDDIT",
    "TARGET_SUBREDDIT",
    "Raw_Title",
    "Raw_Content",
    "combined_text",
    "LINK_SENTIMENT",
    "is_negative_link",
    "Content_Sentiment",
    "Content_Score",
    "LIWC_Anger",
    "high_previous_anger_24h",
]

quality = pd.DataFrame({
    "column": key_columns,
    "missing_values": [int(df[column].isna().sum()) for column in key_columns],
    "unique_values": [int(df[column].nunique(dropna=True)) for column in key_columns],
    "dtype": [str(df[column].dtype) for column in key_columns],
})
display(quality)
quality.to_csv(OUTPUT_DIR / "01_key_column_quality.csv", index=False)


,column,missing_values,unique_values,dtype
0,POST_ID,0,49918,object
1,TIMESTAMP,0,47599,datetime64[ns]
2,SOURCE_SUBREDDIT,0,8605,object
3,TARGET_SUBREDDIT,0,6571,object
4,Raw_Title,0,48367,object
5,Raw_Content,0,48861,object
6,combined_text,0,49200,object
7,LINK_SENTIMENT,0,2,int64
8,is_negative_link,0,2,int64
9,Content_Sentiment,0,3,int64


## Rezultat etapu

Zbior jest gotowy do dalszej analizy: ma staly zakres czasu, brak brakow w kluczowych kolumnach i deterministyczny podzial chronologiczny na train/test. Najwazniejsze ryzyko analityczne to niezbalansowanie klasy negatywnej.
